# 1. concat - Concatenate/stack dataframes

what its :

* combines dataframes along rows (axis=0) and columns(axis=1)
* works mostly by index alignment

In [4]:
import pandas as pd

# monthly 1st sales
df_jan = pd.DataFrame({
    "InvoiceNo": ["A001","A002"],
    "Quantity": [5,10]
})

# 2nd month
df_feb = pd.DataFrame({
    "InvoiceNo": ["A003","A004"],
    "Quantity": [8,12]
})

# Extra columnar data
df_extra = pd.DataFrame({
    "Discount": [0.1, 0.2],
    "Region": ["North","South"]
})

In [5]:
df_jan

,InvoiceNo,Quantity
0,A001,5
1,A002,10


In [6]:
df_feb

,InvoiceNo,Quantity
0,A003,8
1,A004,12


In [7]:
df_extra

,Discount,Region
0,0.1,North
1,0.2,South


In [8]:
# 1. Default vertical concatenation axis = 0
df_all = pd.concat([df_jan, df_feb])
df_all

# index not reset by default - duplicates will remain

,InvoiceNo,Quantity
0,A001,5
1,A002,10
0,A003,8
1,A004,12


In [9]:
# 1.b vertical concatenation with index reset
df_all_reset = pd.concat([df_jan, df_feb], ignore_index = True)
df_all_reset

,InvoiceNo,Quantity
0,A001,5
1,A002,10
2,A003,8
3,A004,12


In [10]:
# 1.b* vertical concatenation with index reset (alternative)
df_all_reset_alt = pd.concat([df_jan, df_feb]).reset_index(drop=True)
df_all_reset_alt

,InvoiceNo,Quantity
0,A001,5
1,A002,10
2,A003,8
3,A004,12


In [11]:
# 1.c Horizontal concatenation (axis = 1)
df_horizontal = pd.concat([df_jan, df_extra], axis=1)
df_horizontal

# rows aligned by index 0 of df_jan -> 0 of df_extra
# 1 of df_jan -> 1 of df_extra

,InvoiceNo,Quantity,Discount,Region
0,A001,5,0.1,North
1,A002,10,0.2,South


In [12]:
help(pd.concat)

Help on function concat in module pandas.core.reshape.concat:

concat(objs: 'Iterable[Series | DataFrame] | Mapping[HashableT, Series | DataFrame]', *, axis: 'Axis' = 0, join: 'str' = 'outer', ignore_index: 'bool' = False, keys: 'Iterable[Hashable] | None' = None, levels=None, names: 'list[HashableT] | None' = None, verify_integrity: 'bool' = False, sort: 'bool' = False, copy: 'bool | None' = None) -> 'DataFrame | Series'
    Concatenate pandas objects along a particular axis.
    
    Allows optional set logic along the other axes.
    
    Can also add a layer of hierarchical indexing on the concatenation axis,
    which may be useful if the labels are the same (or overlapping) on
    the passed axis number.
    
    Parameters
    ----------
    objs : a sequence or mapping of Series or DataFrame objects
        If a mapping is passed, the sorted keys will be used as the `keys`
        argument, unless it is passed, in which case the values will be
        selected (see below). Any 

# 2. merge - sql like joins on columns

what its:

* combine dataframes using key columns
* by default all matching columns will be used as merge key
* similar to sql join (inner, left, right, outer)

In [13]:
df_sales = pd.DataFrame({
    "StockCode": ["P001","P002","P003"],
    "Quantity": [10,5,8]
})

df_products = pd.DataFrame({
    "StockCode" :["P001","P002","P004"],
    "Description": ["Widget","Gadget","Thing"]
})

In [14]:
df_sales

,StockCode,Quantity
0,P001,10
1,P002,5
2,P003,8


In [15]:
df_products

,StockCode,Description
0,P001,Widget
1,P002,Gadget
2,P004,Thing


In [16]:
df_inner = pd.merge(df_products, df_sales, how="inner")
df_inner

,StockCode,Description,Quantity
0,P001,Widget,10
1,P002,Gadget,5


In [17]:
df_left = pd.merge(df_products, df_sales, how="left")
df_left

,StockCode,Description,Quantity
0,P001,Widget,10.0
1,P002,Gadget,5.0
2,P004,Thing,NaN


In [18]:
df_right = pd.merge(df_products, df_sales, how="right")
df_right

,StockCode,Description,Quantity
0,P001,Widget,10
1,P002,Gadget,5
2,P003,NaN,8


In [19]:
df_outer = pd.merge(df_products, df_sales, how="outer")
df_outer

,StockCode,Description,Quantity
0,P001,Widget,10.0
1,P002,Gadget,5.0
2,P003,NaN,8.0
3,P004,Thing,NaN


### How to join with multiple matching column names:

`pd.merge` will match on all matching column names

e.g.
* df1: id, name
* df2: id, name

`on='id'` => `on` is used when you have more than one common column for joining the tables

In [20]:
df1 = pd.DataFrame({'id':[1,2], 'value':[10,20]})
df2 = pd.DataFrame({'id':[1,2], 'value':[20,20]})
merged = pd.merge(df1, df2)
merged

,id,value
0,2,20


In [40]:
merged = pd.merge(df1, df2, on="id")
merged

,id,value_x,value_y
0,1,10,20
1,2,20,20


In [21]:
df_sales = pd.DataFrame({
    "StockCode": ["P001","P002","P003"],
    "owner": ["Markus","Anna","John"],
    "Quantity": [10,5,8]
})

df_products = pd.DataFrame({
    "StockCode" :["P001","P002","P004"],
    "owner": ["Bob", "Anna", "Alice"],
    "Description": ["Widget","Gadget","Thing"]
})

In [22]:
df_inner = pd.merge(df_products, df_sales, how="inner")
df_inner

,StockCode,owner,Description,Quantity
0,P002,Anna,Gadget,5


In [23]:
df_inner = pd.merge(df_products, df_sales, how="inner", on="StockCode")
df_inner

,StockCode,owner_x,Description,owner_y,Quantity
0,P001,Bob,Widget,Markus,10
1,P002,Anna,Gadget,Anna,5


In [24]:
df_left = pd.merge(df_products, df_sales, how="left")
df_left

,StockCode,owner,Description,Quantity
0,P001,Bob,Widget,NaN
1,P002,Anna,Gadget,5.0
2,P004,Alice,Thing,NaN


In [25]:
df_left = pd.merge(df_products, df_sales, how="left", on="StockCode")
df_left

,StockCode,owner_x,Description,owner_y,Quantity
0,P001,Bob,Widget,Markus,10.0
1,P002,Anna,Gadget,Anna,5.0
2,P004,Alice,Thing,NaN,NaN


In [26]:
df_right = pd.merge(df_products, df_sales, how="right")
df_right

,StockCode,owner,Description,Quantity
0,P001,Markus,NaN,10
1,P002,Anna,Gadget,5
2,P003,John,NaN,8


In [27]:
df_right = pd.merge(df_products, df_sales, how="right", on="StockCode")
df_right

,StockCode,owner_x,Description,owner_y,Quantity
0,P001,Bob,Widget,Markus,10
1,P002,Anna,Gadget,Anna,5
2,P003,NaN,NaN,John,8


In [28]:
df_outer = pd.merge(df_products, df_sales, how="outer")
df_outer

,StockCode,owner,Description,Quantity
0,P001,Bob,Widget,NaN
1,P001,Markus,NaN,10.0
2,P002,Anna,Gadget,5.0
3,P003,John,NaN,8.0
4,P004,Alice,Thing,NaN


In [29]:
df_outer = pd.merge(df_products, df_sales, how="outer", on="StockCode")
df_outer

,StockCode,owner_x,Description,owner_y,Quantity
0,P001,Bob,Widget,Markus,10.0
1,P002,Anna,Gadget,Anna,5.0
2,P003,NaN,NaN,John,8.0
3,P004,Alice,Thing,NaN,NaN


# 3. join - index -based joins

what its
* combines dataframes by index (row labels) by default
* can also join on a key by setting the index

In [30]:
df_sales = pd.DataFrame({
    "StockCode": ["P001","P002","P003"],
    "Quantity": [10,5,8]
})

df_products = pd.DataFrame({
    "StockCode" :["P001","P002","P004"],
    "Description": ["Widget","Gadget","Thing"]
})

In [31]:
df_sales_indexed = df_sales.set_index('StockCode')
df_products_indexed = df_products.set_index('StockCode')

In [32]:
df_products_indexed

,Description
StockCode,
P001,Widget
P002,Gadget
P004,Thing


In [33]:
df_sales_indexed

,Quantity
StockCode,
P001,10
P002,5
P003,8


In [34]:
# 1. left join on the basis of index
df_joined_left = df_sales_indexed.join(df_products_indexed, how = "left")
df_joined_left

,Quantity,Description
StockCode,,
P001,10,Widget
P002,5,Gadget
P003,8,NaN


In [35]:
# 1. left join on the basis of index
df_joined_left = df_products_indexed.join(df_sales_indexed, how = "left")
df_joined_left

,Description,Quantity
StockCode,,
P001,Widget,10.0
P002,Gadget,5.0
P004,Thing,NaN


In [36]:
# 1. right join on the basis of index
df_joined_right = df_products_indexed.join(df_sales_indexed, how = "right")
df_joined_right

,Description,Quantity
StockCode,,
P001,Widget,10
P002,Gadget,5
P003,NaN,8


In [37]:
# 1. inner join on the basis of index
df_joined_inner = df_products_indexed.join(df_sales_indexed, how = "inner")
df_joined_inner

,Description,Quantity
StockCode,,
P001,Widget,10
P002,Gadget,5


In [38]:
# 1. outer join on the basis of index
df_joined_outer = df_products_indexed.join(df_sales_indexed, how = "outer")
df_joined_outer

,Description,Quantity
StockCode,,
P001,Widget,10.0
P002,Gadget,5.0
P003,NaN,8.0
P004,Thing,NaN


In [39]:
help(pd.merge)

Help on function merge in module pandas.core.reshape.merge:

merge(left: 'DataFrame | Series', right: 'DataFrame | Series', how: 'MergeHow' = 'inner', on: 'IndexLabel | AnyArrayLike | None' = None, left_on: 'IndexLabel | AnyArrayLike | None' = None, right_on: 'IndexLabel | AnyArrayLike | None' = None, left_index: 'bool' = False, right_index: 'bool' = False, sort: 'bool' = False, suffixes: 'Suffixes' = ('_x', '_y'), copy: 'bool | None' = None, indicator: 'str | bool' = False, validate: 'str | None' = None) -> 'DataFrame'
    Merge DataFrame or named Series objects with a database-style join.
    
    A named Series object is treated as a DataFrame with a single named column.
    
    The join is done on columns or indexes. If joining columns on
    columns, the DataFrame indexes *will be ignored*. Otherwise if joining indexes
    on indexes or indexes on a column or columns, the index will be passed on.
    When performing a cross merge, no column specifications to merge on are
    allo

# Performance considerations

https://python.plainenglish.io/optimizing-pandas-merge-vs-join-for-faster-data-processing-3bfe8bb12aea

In [46]:
import pandas as pd

# Create two example DataFrames
df1 = pd.DataFrame({'ID': [1, 2, 3], 'Value_A': ['A', 'B', 'C']})
df2 = pd.DataFrame({'ID': [1, 2, 4], 'Value_B': ['X', 'Y', 'Z']})

# Merge based on the 'ID' column
merged_df = pd.merge(df1, df2, left_on='ID', right_on='ID', how='inner')
print(merged_df)

   ID Value_A Value_B
0   1       A       X
1   2       B       Y


In [47]:
# Set the index before joining
df1.set_index('ID', inplace=True)
df2.set_index('ID', inplace=True)



In [48]:
# Join DataFrames based on their index
joined_df = df1.join(df2, how='inner')
print(joined_df)

   Value_A Value_B
ID                
1        A       X
2        B       Y


In [74]:
import pandas as pd
import numpy as np
import time

# Create two large DataFrames for the experiment
rows = 10_000_000  # Experiment with 1 million rows
df1 = pd.DataFrame({
    'ID': np.arange(1, rows + 1),
    'Value_A': np.random.randint(1, 100, size=rows)
})

df2 = pd.DataFrame({
    'ID': np.arange(1, rows + 1),
    'Value_B': np.random.randint(1, 100, size=rows)
})



In [75]:
# Measure the time taken to perform the merge
start_time = time.time()
merged_df = pd.merge(df1, df2, left_on='ID', right_on='ID', how='inner')
end_time = time.time()

print(f"Merge completed in {end_time - start_time} seconds.")

Merge completed in 0.051103830337524414 seconds.


In [76]:
# Set the index for both DataFrames before performing the join
df1_indexed = df1.set_index('ID')
df2_indexed = df2.set_index('ID')
df1, df1_indexed

(               ID  Value_A
 0               1       44
 1               2       77
 2               3       23
 3               4       71
 4               5       93
 ...           ...      ...
 9999995   9999996       71
 9999996   9999997       20
 9999997   9999998       59
 9999998   9999999       37
 9999999  10000000       84
 
 [10000000 rows x 2 columns],
           Value_A
 ID               
 1              44
 2              77
 3              23
 4              71
 5              93
 ...           ...
 9999996        71
 9999997        20
 9999998        59
 9999999        37
 10000000       84
 
 [10000000 rows x 1 columns])

In [77]:
# Measure the time taken to perform the join
start_time = time.time()
joined_df = df1_indexed.join(df2_indexed, how='inner')
end_time = time.time()

print(f"Join completed in {end_time - start_time} seconds.")

Join completed in 0.04088282585144043 seconds.


## Performance Advantages of Using Indexed Joins

Indexed joins in Pandas are generally faster because they take advantage of the underlying data structure. Indexes are designed for quick lookups, and as a result, joining DataFrames based on their indexes reduces the computational complexity of the operation. This becomes especially important as the size of the DataFrames grows.

The article states 5-7 times faster joins than merges. My repetition here does not show that. 

Possible reasons:
* My setup is flawed
* different env / version of pandas
* 

In [69]:
pd.__version__

'2.3.3'